In [2]:
from aiohttp.web_fileresponse import content_type
# from langgraph.constants import END, START
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from IPython.display import display
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    # temperature=0.7,
    extra_body={
        "thinking": {
            "type": "disabled",
        }
    }
)

class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    content_type: str

def node_a(state: OverAllState) -> OverAllState:
    poem = model.invoke([f"写一首关于{state['topic']}的诗"]).content
    return {
        "poem": poem,
        "topic": state["topic"],
        "joke": state["joke"],
    }

def node_b(state: OverAllState) -> OverAllState:
    joke  = model.invoke([f"写一个关于{state['topic']}的笑话"]).content
    return {
        "joke": joke,
        "topic": state["topic"],
        "poem": state["poem"],
    }

def my_route(state: OverAllState) -> Literal["poem", "joke"]:
    if "诗" in state["content_type"]:
        return "poem"
    else :
        return "joke"




builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)  # 显式命名节点
builder.add_node("node_b", node_b)
builder.add_conditional_edges(START, my_route, path_map={
    "poem": "node_a",
    "joke": "node_b",
})
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)


graph = builder.compile()
res = graph.invoke({"topic": "猴子", "poem":"", "joke":"", "content_type":"笑话"})

print(f"{res}")
display(graph)
print("Done")

KeyError: 'poem'